# Tutorial: Colab T4 HF Qwen3B Training

Audience:
- This notebook is for running the cleaned HF fine-tune path on a GPU notebook runtime, especially Colab T4.

Prerequisites:
- A GPU runtime is attached before you install packages.
- Either the latest cleaned repo is pushed to GitHub, or you upload `live_translate_repo_snapshot_colab.zip` into the runtime root.
- The training bundle zip is available locally or uploaded into the runtime root as `train_bundle_hf_qwen3b_colab.zip`.


## Outline

1. Detect the runtime and define helper utilities.
2. Resolve the project root from a local checkout, an uploaded repo snapshot zip, `git clone`, or a public repo archive.
3. Resolve the training bundle if the checkout does not include datasets and reports.
4. Install GPU training dependencies and verify CUDA.
5. Prepare the HF run configuration in a clean Colab output directory.
6. Leave smoke and full training gated until you explicitly enable them.
7. Package the final adapter for download.


## Step 0 - Upload The Required Archives

If Step 1 says `Project is not ready`, upload these two files into the Colab runtime root (`/content`) and then rerun Step 1 and Step 2.

- `live_translate_repo_snapshot_colab.zip`
- `train_bundle_hf_qwen3b_colab.zip`


In [ ]:
import sys

in_colab = "google.colab" in sys.modules
if in_colab:
    from google.colab import files
    print("Upload live_translate_repo_snapshot_colab.zip and train_bundle_hf_qwen3b_colab.zip")
    uploaded = files.upload()
    print(sorted(uploaded.keys()))
else:
    print("This helper is for Colab. In a local runtime, place the zip files under the runtime root before running Step 1.")


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import urllib.request
import zipfile
from pathlib import Path
from typing import Any

IN_COLAB = "google.colab" in sys.modules

def run(cmd: list[str | Path], cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess[str]:
    parts = [str(part) for part in cmd]
    print("$", " ".join(parts))
    try:
        completed = subprocess.run(
            parts,
            cwd=str(cwd) if cwd else None,
            check=False,
            text=True,
            capture_output=True,
        )
    except FileNotFoundError as exc:
        print(str(exc))
        return subprocess.CompletedProcess(parts, 127, "", str(exc))
    except Exception as exc:
        print(str(exc))
        return subprocess.CompletedProcess(parts, 1, "", str(exc))
    if completed.stdout:
        print(completed.stdout.rstrip())
    if completed.stderr:
        print(completed.stderr.rstrip())
    if check and completed.returncode != 0:
        raise RuntimeError(f"Command failed ({completed.returncode}): {' '.join(parts)}")
    return completed

def stream_command(
    cmd: list[str | Path],
    cwd: Path | None = None,
    *,
    log_path: Path | None = None,
    check: bool = True,
) -> subprocess.CompletedProcess[str]:
    parts = [str(part) for part in cmd]
    print("$", " ".join(parts))
    if log_path is not None:
        log_path.parent.mkdir(parents=True, exist_ok=True)

    handle = log_path.open("a", encoding="utf-8") if log_path is not None else None
    try:
        env = dict(os.environ)
        env["PYTHONUNBUFFERED"] = "1"
        process = subprocess.Popen(
            parts,
            cwd=str(cwd) if cwd else None,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            encoding="utf-8",
            errors="ignore",
            bufsize=1,
            env=env,
        )
        output_chunks: list[str] = []
        assert process.stdout is not None
        for line in process.stdout:
            output_chunks.append(line)
            print(line, end="")
            if handle is not None:
                handle.write(line)
                handle.flush()
        returncode = process.wait()
        completed = subprocess.CompletedProcess(parts, returncode, "".join(output_chunks), "")
    finally:
        if handle is not None:
            handle.close()

    if check and completed.returncode != 0:
        raise RuntimeError(f"Command failed ({completed.returncode}): {' '.join(parts)}")
    return completed

def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / ".git").exists() and (candidate / "scripts" / "train_hf_sft.py").exists():
            return candidate
    return None

def is_valid_project_root(root: Path) -> bool:
    return (
        root.exists()
        and (root / "scripts" / "train_hf_sft.py").exists()
        and (root / "scripts" / "check_train_env.py").exists()
    )

def download_file(url: str, destination: Path) -> bool:
    destination.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urllib.request.urlopen(url, timeout=60) as response, destination.open("wb") as handle:
            shutil.copyfileobj(response, handle)
        return True
    except Exception as exc:
        print(f"Download failed: {exc}")
        return False

def extract_single_root_zip(archive_path: Path, target_dir: Path) -> bool:
    unpack_root = target_dir.parent / f"{target_dir.name}_archive_unpack"
    shutil.rmtree(unpack_root, ignore_errors=True)
    shutil.rmtree(target_dir, ignore_errors=True)
    unpack_root.mkdir(parents=True, exist_ok=True)
    try:
        with zipfile.ZipFile(archive_path) as archive:
            archive.extractall(unpack_root)
    except Exception as exc:
        print(f"Archive extract failed: {exc}")
        return False
    extracted_dirs = [candidate for candidate in unpack_root.iterdir() if candidate.is_dir()]
    if len(extracted_dirs) != 1:
        print(f"Archive layout unexpected under {unpack_root}")
        return False
    shutil.move(str(extracted_dirs[0]), str(target_dir))
    shutil.rmtree(unpack_root, ignore_errors=True)
    return True

def extract_bundle_zip(bundle_zip: Path, target_dir: Path) -> bool:
    shutil.rmtree(target_dir, ignore_errors=True)
    target_dir.mkdir(parents=True, exist_ok=True)
    try:
        with zipfile.ZipFile(bundle_zip) as archive:
            archive.extractall(target_dir)
        return True
    except Exception as exc:
        print(f"Bundle extract failed: {exc}")
        return False

def read_json_safe(path: Path) -> dict[str, Any] | None:
    if not path.exists():
        return None
    try:
        payload = json.loads(path.read_text(encoding="utf-8", errors="ignore"))
    except Exception:
        return None
    return payload if isinstance(payload, dict) else None

def latest_checkpoint_dir(run_dir: Path) -> Path | None:
    checkpoints: list[tuple[int, Path]] = []
    for candidate in run_dir.glob("checkpoint-*"):
        if not candidate.is_dir():
            continue
        try:
            step = int(candidate.name.split("-")[-1])
        except Exception:
            continue
        checkpoints.append((step, candidate))
    if not checkpoints:
        return None
    checkpoints.sort(key=lambda item: item[0])
    return checkpoints[-1][1]

def tail_text(path: Path, max_lines: int = 40) -> str:
    if not path.exists():
        return ""
    lines = path.read_text(encoding="utf-8", errors="ignore").splitlines()
    return "\n".join(lines[-max_lines:])

def summarize_run_status(run_name: str) -> dict[str, Any]:
    run_dir = OUT_DIR / run_name
    log_path = OUT_DIR / f"{run_name}.live.log"
    summary_path = OUT_DIR / f"train_summary.qwen3b_sft_local.{run_name}.json"
    payload: dict[str, Any] = {
        "run_name": run_name,
        "run_dir": str(run_dir),
        "log_path": str(log_path),
        "summary_path": str(summary_path),
        "log_exists": log_path.exists(),
        "summary_exists": summary_path.exists(),
    }

    checkpoint_dir = latest_checkpoint_dir(run_dir)
    if checkpoint_dir is not None:
        payload["latest_checkpoint"] = str(checkpoint_dir)
        trainer_state = read_json_safe(checkpoint_dir / "trainer_state.json")
        if trainer_state is not None:
            payload["global_step"] = trainer_state.get("global_step")
            payload["max_steps"] = trainer_state.get("max_steps")
            payload["epoch"] = trainer_state.get("epoch")
            log_history = trainer_state.get("log_history")
            if isinstance(log_history, list):
                train_logs = [item for item in log_history if isinstance(item, dict) and ("loss" in item or "learning_rate" in item)]
                eval_logs = [item for item in log_history if isinstance(item, dict) and "eval_loss" in item]
                if train_logs:
                    payload["last_train_log"] = train_logs[-1]
                if eval_logs:
                    payload["last_eval_log"] = eval_logs[-1]

    summary_payload = read_json_safe(summary_path)
    if summary_payload is not None:
        payload["final_ok"] = summary_payload.get("ok")
        payload["train_metrics"] = summary_payload.get("train_metrics")
        payload["eval_metrics"] = summary_payload.get("eval_metrics")

    log_tail = tail_text(log_path)
    if log_tail:
        payload["log_tail"] = log_tail
    return payload

LOCAL_REPO_ROOT = find_repo_root(Path.cwd())
PROJECT_READY = False
ASSETS_READY = False
ENV_READY = False
PREP_READY = False
PROJECT_STATUS = "not_checked"
ASSET_STATUS = "not_checked"
print({
    "python": sys.version.split()[0],
    "in_colab": IN_COLAB,
    "cwd": str(Path.cwd()),
    "local_repo_root": str(LOCAL_REPO_ROOT) if LOCAL_REPO_ROOT else None,
})


## Step 1 - Configure the project source

If this notebook is being run from a local checkout, it will reuse that repo automatically.
If not, it will first look for an uploaded repo snapshot zip, then try `git clone`, and finally fall back to downloading the public GitHub archive from `codeload.github.com`.

The public repo gives you code only if the latest training scripts were actually pushed. The dataset and quality report are resolved in the next step because those artifacts are usually not stored in Git.


In [ ]:
REPO_URL = "https://github.com/Memedem1n/live-translate-assistant.git"
REPO_BRANCH = "main"
REPO_ARCHIVE_URL = f"https://codeload.github.com/Memedem1n/live-translate-assistant/zip/refs/heads/{REPO_BRANCH}"
REPO_MODE = "auto"  # auto, existing, snapshot, clone
BUNDLE_MODE = "auto"  # auto, existing, upload, skip
RUNTIME_ROOT = Path("/content") if IN_COLAB else Path.cwd()
EXISTING_PROJECT_ROOT = LOCAL_REPO_ROOT or (RUNTIME_ROOT / "live-translate-assistant")
CLONE_PROJECT_ROOT = RUNTIME_ROOT / "live-translate-assistant"
REPO_ARCHIVE_PATH = RUNTIME_ROOT / "live-translate-assistant-main.zip"
EXISTING_REPO_SNAPSHOT_ZIP = (LOCAL_REPO_ROOT / "artifacts" / "live_translate_repo_snapshot_colab.zip") if LOCAL_REPO_ROOT else (RUNTIME_ROOT / "live_translate_repo_snapshot_colab.zip")
UPLOADED_REPO_SNAPSHOT_ZIP = RUNTIME_ROOT / "live_translate_repo_snapshot_colab.zip"
EXISTING_BUNDLE_ZIP = (LOCAL_REPO_ROOT / "artifacts" / "train_bundle_hf_qwen3b_colab.zip") if LOCAL_REPO_ROOT else (RUNTIME_ROOT / "train_bundle_hf_qwen3b_colab.zip")
UPLOADED_BUNDLE_ZIP = RUNTIME_ROOT / "train_bundle_hf_qwen3b_colab.zip"
PROJECT_ROOT = EXISTING_PROJECT_ROOT if REPO_MODE == "existing" else (LOCAL_REPO_ROOT or CLONE_PROJECT_ROOT)
OUT_DIR = PROJECT_ROOT / "artifacts" / "finetune" / "hf_qwen3b_colab"
ATTEMPTS_PATH = PROJECT_ROOT / "artifacts" / "finetune" / "training_attempts.hf_qwen3b.colab.json"
TRAIN_BUNDLE_DIR = RUNTIME_ROOT / "train_bundle_runtime"
DATASET = PROJECT_ROOT / "artifacts" / "finetune" / "interview_train.sft.v2.jsonl"
VALID_DATASET = PROJECT_ROOT / "artifacts" / "finetune" / "interview_train.sft.v2.valid.jsonl"
QUALITY_REPORT = PROJECT_ROOT / "artifacts" / "finetune" / "interview_quality_report.v2.json"
HF_TOKEN_PRESENT = bool(os.environ.get("HF_TOKEN"))

print({
    "project_root": str(PROJECT_ROOT),
    "repo_url": REPO_URL,
    "repo_branch": REPO_BRANCH,
    "repo_archive_url": REPO_ARCHIVE_URL,
    "repo_snapshot_candidates": [str(EXISTING_REPO_SNAPSHOT_ZIP), str(UPLOADED_REPO_SNAPSHOT_ZIP)],
    "bundle_mode": BUNDLE_MODE,
    "bundle_candidates": [str(EXISTING_BUNDLE_ZIP), str(UPLOADED_BUNDLE_ZIP)],
    "out_dir": str(OUT_DIR),
    "attempts_path": str(ATTEMPTS_PATH),
    "hf_token_present": HF_TOKEN_PRESENT,
})


In [ ]:
PROJECT_READY = False
PROJECT_STATUS = "not_ready"

clone_target = CLONE_PROJECT_ROOT
repo_snapshot_candidates: list[Path] = []
if REPO_MODE in {"auto", "snapshot"}:
    repo_snapshot_candidates.extend([EXISTING_REPO_SNAPSHOT_ZIP, UPLOADED_REPO_SNAPSHOT_ZIP])

if REPO_MODE in {"auto", "existing"} and is_valid_project_root(PROJECT_ROOT):
    PROJECT_READY = True
    PROJECT_STATUS = "existing_repo"
else:
    repo_snapshot_zip = next((candidate for candidate in repo_snapshot_candidates if candidate.exists()), None)
    if repo_snapshot_zip is not None:
        print(f"Using repo snapshot: {repo_snapshot_zip}")
        if extract_single_root_zip(repo_snapshot_zip, clone_target) and is_valid_project_root(clone_target):
            PROJECT_ROOT = clone_target
            PROJECT_READY = True
            PROJECT_STATUS = "snapshot_zip"

    if not PROJECT_READY and REPO_MODE in {"auto", "clone"}:
        clone_commands: list[list[str | Path]] = [
            ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, clone_target],
            ["git", "clone", "--depth", "1", REPO_URL, clone_target],
        ]
        for command in clone_commands:
            if clone_target.exists() and not is_valid_project_root(clone_target):
                shutil.rmtree(clone_target, ignore_errors=True)
            completed = run(command, check=False)
            if completed.returncode == 0 and is_valid_project_root(clone_target):
                PROJECT_ROOT = clone_target
                PROJECT_READY = True
                PROJECT_STATUS = "cloned_repo"
                break

    if not PROJECT_READY and REPO_MODE in {"auto", "clone"}:
        print("Trying public repo archive fallback...")
        archive_downloaded = download_file(REPO_ARCHIVE_URL, REPO_ARCHIVE_PATH)
        archive_extracted = archive_downloaded and extract_single_root_zip(REPO_ARCHIVE_PATH, clone_target)
        archive_valid = archive_extracted and is_valid_project_root(clone_target)
        print({
            "archive_downloaded": archive_downloaded,
            "archive_extracted": archive_extracted,
            "archive_valid": archive_valid,
        })
        if archive_valid:
            PROJECT_ROOT = clone_target
            PROJECT_READY = True
            PROJECT_STATUS = "downloaded_archive"

if PROJECT_READY:
    os.chdir(PROJECT_ROOT)
    OUT_DIR = PROJECT_ROOT / "artifacts" / "finetune" / "hf_qwen3b_colab"
    ATTEMPTS_PATH = PROJECT_ROOT / "artifacts" / "finetune" / "training_attempts.hf_qwen3b.colab.json"
    DATASET = PROJECT_ROOT / "artifacts" / "finetune" / "interview_train.sft.v2.jsonl"
    VALID_DATASET = PROJECT_ROOT / "artifacts" / "finetune" / "interview_train.sft.v2.valid.jsonl"
    QUALITY_REPORT = PROJECT_ROOT / "artifacts" / "finetune" / "interview_quality_report.v2.json"
    print(json.dumps({
        "project_ready": PROJECT_READY,
        "project_status": PROJECT_STATUS,
        "project_root": str(PROJECT_ROOT),
    }, indent=2))
else:
    PROJECT_STATUS = "repo_unavailable"
    print("Project bootstrap did not complete.")
    print("Most common causes: GitHub has an older snapshot than your local training code, the repo URL is inaccessible, or no repo snapshot zip was uploaded.")
    print("Fix options:")
    print("1. Upload live_translate_repo_snapshot_colab.zip and leave REPO_MODE = 'auto'.")
    print("2. Push the latest cleaned training scripts to the public branch and rerun the cell.")
    print("3. Set REPO_MODE = 'existing' and point EXISTING_PROJECT_ROOT to a mounted repo path.")


## Step 2 - Resolve the training assets

A public repo checkout only gives you the code. This step reuses in-repo artifacts when they exist, otherwise it extracts the exported training bundle zip.


In [ ]:
ASSETS_READY = False
ASSET_STATUS = "not_ready"

repo_dataset = PROJECT_ROOT / "artifacts" / "finetune" / "interview_train.sft.v2.jsonl"
repo_valid_dataset = PROJECT_ROOT / "artifacts" / "finetune" / "interview_train.sft.v2.valid.jsonl"
repo_quality_report = PROJECT_ROOT / "artifacts" / "finetune" / "interview_quality_report.v2.json"

bundle_candidates: list[Path] = []
if BUNDLE_MODE in {"auto", "existing"}:
    bundle_candidates.append(EXISTING_BUNDLE_ZIP)
if BUNDLE_MODE in {"auto", "upload"} and UPLOADED_BUNDLE_ZIP not in bundle_candidates:
    bundle_candidates.append(UPLOADED_BUNDLE_ZIP)

if not PROJECT_READY:
    print("Project is not ready. Fix the source/bootstrap step first.")
else:
    missing_repo_assets = [
        str(path)
        for path in [repo_dataset, repo_valid_dataset, repo_quality_report]
        if not path.exists()
    ]
    if not missing_repo_assets:
        DATASET = repo_dataset
        VALID_DATASET = repo_valid_dataset
        QUALITY_REPORT = repo_quality_report
        ASSETS_READY = True
        ASSET_STATUS = "repo_artifacts"
    else:
        print("Repo checkout is missing training assets.")
        print(json.dumps(missing_repo_assets, indent=2))
        if BUNDLE_MODE == "skip":
            print("BUNDLE_MODE = 'skip', so notebook will not try to resolve the uploaded bundle.")
        else:
            bundle_zip = next((candidate for candidate in bundle_candidates if candidate.exists()), None)
            if bundle_zip is None:
                print("Training bundle not found. Upload train_bundle_hf_qwen3b_colab.zip to the runtime root and rerun this cell.")
                print("Expected bundle locations:")
                print(json.dumps([str(path) for path in bundle_candidates], indent=2))
            else:
                print(f"Using training bundle: {bundle_zip}")
                if extract_bundle_zip(bundle_zip, TRAIN_BUNDLE_DIR):
                    DATASET = TRAIN_BUNDLE_DIR / "interview_train.sft.v2.jsonl"
                    VALID_DATASET = TRAIN_BUNDLE_DIR / "interview_train.sft.v2.valid.jsonl"
                    QUALITY_REPORT = TRAIN_BUNDLE_DIR / "interview_quality_report.v2.json"
                    missing_bundle_assets = [
                        str(path)
                        for path in [DATASET, VALID_DATASET, QUALITY_REPORT]
                        if not path.exists()
                    ]
                    if missing_bundle_assets:
                        ASSET_STATUS = "bundle_missing_files"
                        print("Bundle extracted, but required files are missing:")
                        print(json.dumps(missing_bundle_assets, indent=2))
                    else:
                        ASSETS_READY = True
                        ASSET_STATUS = f"bundle:{bundle_zip.name}"

if ASSETS_READY:
    print(json.dumps({
        "asset_status": ASSET_STATUS,
        "dataset": str(DATASET),
        "valid_dataset": str(VALID_DATASET),
        "quality_report": str(QUALITY_REPORT),
        "out_dir": str(OUT_DIR),
        "attempts_path": str(ATTEMPTS_PATH),
    }, indent=2))
else:
    print("Training assets are not ready yet.")


## Step 3 - Verify GPU runtime and install training dependencies

This step keeps the runtime simple: it uses the active notebook Python instead of the Windows PowerShell wrappers.
On Colab, `torch` is usually already present with CUDA support, so the cell only installs it if CUDA is not available.


In [ ]:
ENV_READY = False
if not PROJECT_READY:
    print("Project is not ready. Fix the source/bootstrap step first.")
elif not ASSETS_READY:
    print("Training assets are not ready. Run the bundle cell first.")
else:
    gpu_probe = run(["nvidia-smi"], check=False)
    if gpu_probe.returncode != 0:
        print("nvidia-smi failed. Check that the runtime is actually attached to a GPU.")

    torch_ok = False
    HF_DEPS = [
        "accelerate==1.13.0",
        "datasets==4.6.1",
        "peft==0.18.1",
        "sentencepiece==0.2.1",
        "safetensors==0.7.0",
        "transformers==5.3.0",
        "trl==0.29.0",
        "bitsandbytes==0.49.2",
    ]

    try:
        import torch
        torch_ok = bool(torch.cuda.is_available())
        print({
            "torch": torch.__version__,
            "cuda_available": torch.cuda.is_available(),
            "cuda_device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        })
    except Exception as exc:
        print("Initial torch probe failed:", exc)

    torch_install_ok = True
    if not torch_ok:
        torch_install = run([sys.executable, "-m", "pip", "install", "-U", "torch", "torchvision", "torchaudio"], check=False)
        torch_install_ok = torch_install.returncode == 0

    deps_install = run([sys.executable, "-m", "pip", "install", "-U", *HF_DEPS], check=False)
    env_probe = run([sys.executable, "scripts/check_train_env.py", "--backend", "hf"], cwd=PROJECT_ROOT, check=False)

    ENV_READY = torch_install_ok and deps_install.returncode == 0 and env_probe.returncode == 0
    if ENV_READY:
        print("Training environment ready.")
    else:
        print("Training environment is not ready yet. Review the command output above, fix the issue, then rerun this cell.")


## Step 4 - Prepare the Colab run directory

This creates a fresh HF prep directory under `artifacts/finetune/hf_qwen3b_colab`, writes the smoke/full config files, and validates tokenization before training.


In [ ]:
PREP_READY = False
SMOKE_CONFIG_PATH = OUT_DIR / "train_config.qwen3b_sft_local.smoke.json"
FULL_CONFIG_PATH = OUT_DIR / "train_config.qwen3b_sft_local.full.json"
prepare_cmd = [
    sys.executable,
    "scripts/train_hf_sft.py",
    "--dataset",
    str(DATASET),
    "--valid-dataset",
    str(VALID_DATASET),
    "--out-dir",
    str(OUT_DIR),
    "--attempts-path",
    str(ATTEMPTS_PATH),
    "--profile",
    "qwen3b_sft_local",
    "--quality-report",
    str(QUALITY_REPORT),
    "--require-quality-pass",
    "--train-env",
    "colab",
]

if not PROJECT_READY:
    print("Project is not ready. Fix the source/bootstrap step first.")
elif not ASSETS_READY:
    print("Training assets are not ready. Run the bundle cell first.")
elif not ENV_READY:
    print("Environment is not ready. Run the dependency check cell first.")
else:
    prepare_result = run(prepare_cmd, cwd=PROJECT_ROOT, check=False)
    if prepare_result.returncode == 0 and ATTEMPTS_PATH.exists():
        prepare_payload = json.loads(ATTEMPTS_PATH.read_text(encoding="utf-8"))
        PREP_READY = bool(prepare_payload.get("ok"))
        generated = prepare_payload.get("generated", {})
        SMOKE_CONFIG_PATH = Path(str(generated.get("smoke_config") or SMOKE_CONFIG_PATH))
        FULL_CONFIG_PATH = Path(str(generated.get("full_config") or FULL_CONFIG_PATH))
        print({
            "ok": prepare_payload["ok"],
            "train_records": prepare_payload["train_records"],
            "valid_records": prepare_payload["valid_records"],
            "smoke_config": str(SMOKE_CONFIG_PATH),
            "full_config": str(FULL_CONFIG_PATH),
        })
    else:
        print("Prepare step failed. Review the command output above, then rerun this cell.")


## Step 5 - Smoke run

This cell now streams the live trainer output into the notebook and also writes a persistent log file under `hf_qwen3b_colab/smoke.live.log`.
If your editor disconnects, reopen the notebook and use the status snapshot cell below to inspect the latest checkpoint and metrics.


In [ ]:
RUN_SMOKE = False
smoke_summary_path = OUT_DIR / "train_summary.qwen3b_sft_local.smoke.json"
smoke_log_path = OUT_DIR / "smoke.live.log"
smoke_cmd = [
    sys.executable,
    "scripts/train_hf_sft.py",
    "--config",
    str(SMOKE_CONFIG_PATH),
    "--execute",
]

if not PREP_READY:
    print("Prepare step is not ready yet. Run the prepare cell successfully first.")
elif RUN_SMOKE:
    print(json.dumps({
        "smoke_config": str(SMOKE_CONFIG_PATH),
        "smoke_log_path": str(smoke_log_path),
        "smoke_run_dir": str(OUT_DIR / "smoke"),
    }, indent=2))
    smoke_result = stream_command(smoke_cmd, cwd=PROJECT_ROOT, log_path=smoke_log_path, check=False)
    if smoke_result.returncode != 0:
        print("Smoke training failed. Review the live output or the log file before retrying.")
else:
    print("Set RUN_SMOKE = True and re-run this cell when you want to start smoke training.")

print(json.dumps(summarize_run_status("smoke"), indent=2))


## Step 6 - Full training

This cell also streams live trainer output and writes it to `hf_qwen3b_colab/full.live.log`.
If VS Code loses the remote Jupyter connection, the Colab runtime often keeps training; reconnect and rerun the status snapshot cell to inspect the latest step, epoch, train loss, eval loss, and recent log tail.


In [ ]:
RUN_FULL = False
smoke_summary_path = OUT_DIR / "train_summary.qwen3b_sft_local.smoke.json"
full_log_path = OUT_DIR / "full.live.log"
full_cmd = [
    sys.executable,
    "scripts/train_hf_sft.py",
    "--config",
    str(FULL_CONFIG_PATH),
    "--execute",
]

if not PREP_READY:
    print("Prepare step is not ready yet. Run the prepare cell successfully first.")
elif RUN_FULL:
    if not smoke_summary_path.exists():
        print("Smoke summary missing. Run smoke successfully before full training.")
    else:
        print(json.dumps({
            "full_config": str(FULL_CONFIG_PATH),
            "full_log_path": str(full_log_path),
            "full_run_dir": str(OUT_DIR / "full"),
        }, indent=2))
        full_result = stream_command(full_cmd, cwd=PROJECT_ROOT, log_path=full_log_path, check=False)
        if full_result.returncode != 0:
            print("Full training failed. Review the live output or the log file before retrying.")
else:
    print("Set RUN_FULL = True and re-run this cell after a clean smoke pass.")


In [ ]:
WATCH_RUN = "full"  # full or smoke
snapshot = summarize_run_status(WATCH_RUN)
print(json.dumps({k: v for k, v in snapshot.items() if k != "log_tail"}, indent=2))
if snapshot.get("log_tail"):
    print("\n--- recent log tail ---")
    print(snapshot["log_tail"])


## Step 7 - Package and export the result

Once full training finishes, this cell writes the Ollama `Modelfile`, zips the full adapter directory, and exposes the archive path for download.


In [ ]:
adapter_dir = OUT_DIR / "full"
adapter_file = adapter_dir / "adapter_model.safetensors"

if adapter_file.exists():
    run([
        sys.executable,
        "scripts/package_lora_for_ollama.py",
        "--base",
        "qwen2.5:3b-instruct-q4_K_M",
        "--adapter-dir",
        str(adapter_dir),
        "--model-name",
        "interview-copilot-hf-qwen3b-colab",
    ], cwd=PROJECT_ROOT)

    archive_base = PROJECT_ROOT / "artifacts" / "finetune" / "hf_qwen3b_colab_full"
    archive_path = archive_base.with_suffix(".zip")
    if archive_path.exists():
        archive_path.unlink()
    shutil.make_archive(str(archive_base), "zip", adapter_dir)

    print({
        "modelfile": str(adapter_dir / "Modelfile"),
        "zip": str(archive_path),
    })

    if IN_COLAB:
        from google.colab import files
        print("Use files.download(...) below if you want to pull the zip locally.")
        # files.download(str(archive_path))
else:
    print("Full adapter not found yet. Finish the full run before packaging.")


## Pitfalls

- If `nvidia-smi` does not show a T4 or another CUDA-capable GPU, stop and change the runtime before installing packages.
- A public repo checkout gives you only what was actually pushed. If the latest training scripts are local-only, upload `live_translate_repo_snapshot_colab.zip`.
- Keep `train_bundle_hf_qwen3b_colab.zip` handy because the dataset and quality artifacts are not guaranteed to be in Git.
- First model download can take several minutes. Setting `HF_TOKEN` speeds that up but is not strictly required.
- Keep smoke and full outputs separate from local Windows outputs by using the dedicated `hf_qwen3b_colab` directory in this notebook.
